# 📊 SVD for Image Compression and Denoising

Interactive demo using Singular Value Decomposition (SVD) on a real image.

- Image compression
- Noise reduction
- PSNR, Compression Ratio, Energy metrics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, util
from skimage.metrics import peak_signal_noise_ratio

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['image.cmap'] = 'gray'

## 1. Load Image

In [ ]:
img = data.camera()[:600, :800] / 255.0
m, n = img.shape
print(f"Shape: {m} × {n}")

plt.imshow(img)
plt.title('Original Image')
plt.axis('off')
plt.show()

## 2. Add Noise

In [ ]:
noise_sigma = 0.1
img_noisy = util.random_noise(img, mode='gaussian', var=noise_sigma**2)
img_noisy = np.clip(img_noisy, 0, 1)

psnr_noisy = peak_signal_noise_ratio(img, img_noisy, data_range=1.0)

plt.imshow(img_noisy)
plt.title(f'Noisy Image (PSNR = {psnr_noisy:.2f} dB)')
plt.axis('off')
plt.show()

## 3. Compute SVD

In [ ]:
U, Sigma, Vt = np.linalg.svd(img_noisy, full_matrices=False)

plt.semilogy(Sigma, 'b-', linewidth=2)
plt.title('Singular Values (log scale)')
plt.xlabel('Index')
plt.ylabel('Singular Value')
plt.grid(True, alpha=0.3)
plt.show()

## 4. Denoising (k=50)

In [ ]:
k = 50
img_denoised = U[:, :k] @ np.diag(Sigma[:k]) @ Vt[:k, :]
img_denoised = np.clip(img_denoised, 0, 1)

psnr_denoised = peak_signal_noise_ratio(img, img_denoised, data_range=1.0)

plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1); plt.imshow(img); plt.title('Original'); plt.axis('off')
plt.subplot(1, 3, 2); plt.imshow(img_noisy); plt.title(f'Noisy'); plt.axis('off')
plt.subplot(1, 3, 3); plt.imshow(img_denoised); plt.title(f'Denoised (k={k})'); plt.axis('off')
plt.suptitle(f'SVD Denoising (PSNR = {psnr_denoised:.2f} dB)', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Compression (k=30)

In [ ]:
k_comp = 30
U_c, Sigma_c, Vt_c = np.linalg.svd(img, full_matrices=False)
img_compressed = U_c[:, :k_comp] @ np.diag(Sigma_c[:k_comp]) @ Vt_c[:k_comp, :]

psnr_comp = peak_signal_noise_ratio(img, img_compressed, data_range=1.0)
energy = np.sum(Sigma_c[:k_comp]**2) / np.sum(Sigma_c**2)
cr = (m * n) / (k_comp * (m + n + 1))

plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1); plt.imshow(img); plt.title('Original'); plt.axis('off')
plt.subplot(1, 3, 2); plt.imshow(img_compressed); plt.title(f'Compressed (k={k_comp})'); plt.axis('off')
plt.subplot(1, 3, 3); plt.bar(['Full', 'Compressed'], [m*n, k_comp*(m+n+1)]); plt.title(f'CR = {cr:.2f}x'); plt.ylabel('Values')
plt.suptitle(f'Compression: PSNR={psnr_comp:.2f} dB, Energy={energy:.1%}', fontsize=14)
plt.tight_layout()
plt.show()